# Lab 13: In-Context LLM Prompting & Retrieval-Augmented Generation (RAG) Systems

Welcome to Laboratory 13! In this lab, we build modern generative AI architectures:
1. **Prompt Engineering Paradigms**: System Prompting, Few-Shot Demonstrations, and Chain-of-Thought (CoT) reasoning.
2. **Dense Vector Store & Retrieval**: Implement vector document indexing with cosine similarity search.
3. **End-to-End RAG Pipeline**: Ground generative language models with external factual knowledge to prevent hallucinations.


## 1. Technical Preliminaries & Environment Configuration


In [ ]:
# Import numerical and text processing libraries
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

print('Vector indexing and retrieval libraries ready!')


## 2. In-Memory Vector Store & Document Retrieval Pipeline

### Conceptual Overview: Retrieval-Augmented Generation (RAG) Architecture
Standard LLMs hallucinate when asked about domain-specific or private documents. RAG resolves this through a three-stage pipeline:
1. **Indexing**: Chunk documents and project them into normalized continuous vector embeddings.
2. **Retrieval**: Given a user query $\mathbf{q}$, compute cosine similarities $\mathbf{s} = \mathbf{D} \mathbf{q}$ and retrieve top-$k$ relevant passages.
3. **Augmented Generation**: Construct an augmented prompt combining the retrieved context with the user query:
   $$\text{Prompt} = \text{"[Context: }" + \text{Passage} + \text{"] Question: }" + \text{Query}$$


### Helper Class: `InMemoryVectorStore`
The class below handles document embedding, normalization, and top-$k$ nearest neighbor retrieval.


In [ ]:
class InMemoryVectorStore:
    """Vector store that indexes documents and retrieves top-k passages via cosine similarity."""
    def __init__(self, documents: list):
        self.documents = documents
        self.vectorizer = TfidfVectorizer()
        
        # Extract and normalize document embeddings to unit length (L2 norm = 1.0)
        raw_embeddings = self.vectorizer.fit_transform([d['text'] for d in documents]).toarray()
        norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
        self.doc_embeddings = raw_embeddings / np.maximum(norms, 1e-8)
        
    def retrieve(self, query_str: str, top_k: int = 1):
        """Retrieves the most semantically relevant document for a given query string."""
        # Transform query string into normalized vector embedding
        raw_q = self.vectorizer.transform([query_str]).toarray()
        q_norm = np.linalg.norm(raw_q)
        q_vec = raw_q / (q_norm + 1e-8)
        
        # Compute dot products (cosine similarities between query and all stored documents)
        similarity_scores = (self.doc_embeddings @ q_vec.T).squeeze()
        
        # Retrieve highest scoring document index
        top_idx = int(np.argmax(similarity_scores))
        best_doc = self.documents[top_idx]
        best_score = float(similarity_scores[top_idx])
        return best_doc, best_score

# Knowledge base of domain documents
knowledge_base = [
    {'title': 'PyTorch Tensors', 'text': 'PyTorch tensors are multi-dimensional arrays supporting GPU acceleration and autograd.'},
    {'title': 'ResNet Architecture', 'text': 'ResNet introduces residual skip connections to allow training of extremely deep networks.'},
    {'title': 'LoRA Fine-Tuning', 'text': 'LoRA freezes base model weights and trains low-rank decomposition matrices for parameter efficiency.'},
    {'title': 'Diffusion Models', 'text': 'Diffusion models generate images by gradually removing noise in a learned reverse Markov process.'}
]

# Instantiate Vector Store
vector_store = InMemoryVectorStore(knowledge_base)

# Test Retrieval
query = 'How does LoRA make fine-tuning parameter efficient?'
matched_doc, similarity = vector_store.retrieve(query)

print(f'User Query: "{query}"')
print(f'Retrieved Match: [{matched_doc["title"]}] (Cosine Score: {similarity:.4f})')
print(f'Passage Content: {matched_doc["text"]}')


### Prompt Augmentation Pipeline Function: `generate_rag_prompt`
The function below formats retrieved knowledge into a grounded prompt for generative language models.


In [ ]:
def generate_rag_prompt(user_query: str, vector_store: InMemoryVectorStore) -> str:
    """Constructs a grounded RAG prompt containing retrieved factual context."""
    retrieved_doc, score = vector_store.retrieve(user_query)
    
    rag_prompt = f"""### System:
You are an expert AI assistant. Answer the user question using ONLY the verified context provided below.

### Verified Context:
[{retrieved_doc['title']}]: {retrieved_doc['text']}

### User Question:
{user_query}

### Grounded Answer:"""
    return rag_prompt

# Display synthesized RAG prompt ready for LLM consumption
sample_prompt = generate_rag_prompt('What are PyTorch tensors used for?', vector_store)
print('Synthesized Grounded RAG Prompt:\n')
print(sample_prompt)


## 3. Summary & Key Takeaways
1. **In-Context Prompting**: Steering foundation models through zero-shot, few-shot, and CoT prompts without modifying underlying model weights.
2. **Retrieval-Augmented Generation (RAG)**: Dynamically injects factual, up-to-date domain knowledge into prompts, eliminating model hallucinations.
3. **Vector Similarity**: Normalizing embeddings allows fast dot-product cosine similarity search across massive knowledge repositories.
